![DB Academy](./Includes/images/db-academy.png)

# PITR & Snapshots

---

This notebook provides the foundational knowledge behind Point in Time Recovery and Snapshots - 2 key features of Lakebase.

## Learning Objectives
By the end of this lecture, you will understand:
1. **Point-in-Time Recovery (PITR)**: What it is? How to properly set up a restore window. We will also dive into how to restore, what happens after a restore operation, and finally when to use it.
2. **Snapshots**: What it is? How to set up a snapshot schedule and when to use this feature!

---
## A. Point-in-Time Recovery (PITR)

PITR lets you restore a branch to **any exact moment** within a configurable window. It is powered by the same transaction log that Lakebase maintains for all root branches — no extra setup required.

---

### What is a Restore Window?

The **restore window** is how far back in time you can recover. It is configurable from **0 to 30 days** and applies uniformly across *all* branches in the project.

| Setting | Effect |
|---------|--------|
| Longer window (e.g. 30 days) | More recovery flexibility, but higher storage cost |
| Shorter window (e.g. 1 day) | Lower storage cost, but limited recovery range |
| 0 days | PITR is effectively disabled |

> ⚠️ The restore window is a **project-level setting** — you cannot set different windows per branch.

---

### How to Perform a Restore Operation

PITR is performed through the **Lakebase App UI** in three steps:

1. Open your project  →  Backup & Restore
![Backup & Restore UI](Includes/images/pitr/backup_restore_1.png)
2. Select your source branch
![Backup & Restore UI](Includes/images/pitr/backup_restore_2.png)
   - Use the date/time picker to choose your restore point

3. Click "Restore to point in time"  →  Restore
![Backup & Restore UI](Includes/images/pitr/backup_restore_3.png)

---

### What Happens After a Restore Operation?

A restore **never modifies your existing branch**. Instead:

| Outcome | Detail |
|---------|--------|
| **New root branch created** | Contains the full database state from the specified point in time |
| **Original branch unchanged** | Your production branch keeps running without interruption |
| **Existing connections unaffected** | Apps connected to the original branch see no disruption |
| **Manual cutover required** | To use the restored data in production, you update your app's connection string to point to the new branch |

> ⚠️ Projects support a maximum of **3 root branches**. If you are at the limit, you must delete an existing root branch before creating a restore branch.

> 💡 A restore recovers **all databases** within a branch — you cannot restore a single database in isolation.

---

### When to Use PITR

PITR is optimised for **unexpected, unplanned events** where you need to recover to a precise moment:

| Scenario | Example |
|----------|----------|
| ✅ Accidental data deletion | `DELETE FROM orders` without a `WHERE` clause |
| ✅ Destructive schema changes | `DROP TABLE inventory_main` run on the wrong environment |
| ✅ Application bug corruption | A code deploy that wrote bad data for 10 minutes |
| ❌ Planned pre-change backup | Use a **Snapshot** instead — it's more explicit and cheaper |

---
## 📸 B. Snapshots

A snapshot is an **explicit, named point-in-time capture** of a root branch. Unlike the continuous PITR transaction log, snapshots are discrete restore points you create on demand or on a schedule.

---

### What is a Snapshot?

A snapshot captures the **complete state of a root branch** — schema and all data — at the moment it is taken. Key properties:

| Property | Detail |
|----------|--------|
| **Instant creation** | Snapshots are created immediately with minimal performance impact |
| **Root branches only** | Snapshots can only be taken on root (production-level) branches |
| **Manual limit** | Up to **10 manual snapshots** per project |
| **Scheduled snapshots** | Do not count toward the 10-snapshot limit |
| **Deletion is permanent** | Deleted snapshots cannot be recovered |

---

### Create a Snapshot Schedule

Automated snapshots run at regular intervals so you always have a recent restore point without manual effort:

1. Open your project  →  Backup & Restore
2. Click "Edit schedule"
![Backup & Restore UI](Includes/images/pitr/snapshot_1.png)
3. Choose frequency:  Daily  |  Weekly  |  Monthly
![Backup & Restore UI](Includes/images/pitr/snapshot_2.png)
4. Set your retention period  →  Update Schedule
![Backup & Restore UI](Includes/images/pitr/snapshot_3.png)

> ⚠️ When a scheduled snapshot's retention period expires it is **automatically deleted** and cannot be recovered. Size your retention window to match your recovery time objectives.

> 💡 Manual snapshots are **unaffected** by backup schedule retention settings — they persist until you explicitly delete them.

---

### Perform a Restore from a Snapshot

1. Open your project  →  Backup & Restore
2. Find the desired snapshot in the list
3. Click "Restore"  →  Confirm
![Backup & Restore UI](Includes/images/pitr/restore_1.png)

The same non-destructive restore model applies as with PITR:

| Outcome | Detail |
|---------|--------|
| **New root branch created** | Named `branch_from_snapshot_<timestamp>` |
| **Original branch unchanged** | Production continues operating normally |
| **Next steps** | Preview data → rename branch → redirect your application |

---

### When to Use Snapshots

Snapshots are optimised for **planned, proactive backups** where you want a named, explicit restore point:

| Scenario | Example |
|----------|----------|
| ✅ Before risky schema migrations | `ALTER TABLE` that drops columns or changes types |
| ✅ Before a major deployment | Spring Sale go-live cutover |
| ✅ Regular scheduled backups | Daily snapshot at 02:00 UTC as a safety net |
| ✅ Compliance checkpoints | End-of-month data freeze for audit purposes |
| ❌ Recovering from an unknown moment | Use **PITR** — you need granularity beyond snapshot frequency |

---
## C. PITR vs Snapshots — Quick Reference

| | **PITR** | **Snapshots** |
|---|---|---|
| **Best for** | Unexpected events (accidents, bugs) | Planned events (deployments, migrations) |
| **Granularity** | Any second within the restore window | Discrete named points |
| **Window / Limit** | 0 – 30 days (project-wide) | 10 manual + unlimited scheduled |
| **Storage cost** | Increases with longer window | Per-snapshot overhead |
| **Setup required** | None — always on for root branches | Scheduled or manual creation |
| **Restore target** | New root branch | New root branch |
| **Original branch** | Unchanged | Unchanged |
| **Root branch limit** | Max 3 per project | Max 3 per project |

> 💡 **Rule of thumb:** Use Snapshots *before* you make a change. Use PITR *after* something goes wrong.